# Appendix: Prompt Engineering

The single cheapest way to improve an agent is usually not more code — it's a better prompt. This appendix consolidates the prompt techniques scattered through the course (system prompts in 1.1, tool descriptions in 1.2, structured output in 0.2) into one reference, and mirrors appendices D and E of the n8n course.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/appendix_prompt_engineering.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — same openai + OpenRouter client as Block 0. Run once.
!pip install -q openai

import os, getpass
from openai import OpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL = "meta-llama/llama-3.3-70b-instruct"
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

def ask(system, user, temperature=0):
    r = client.chat.completions.create(model=MODEL, temperature=temperature,
        messages=[{"role":"system","content":system},{"role":"user","content":user}])
    return r.choices[0].message.content

print("Ready:", MODEL)

## The system prompt: the agent's job description

The system prompt sets standing behavior for the whole conversation. A good one usually has three parts:

1. **Role** — who the model is acting as ("You are a billing support specialist").
2. **Rules** — explicit constraints ("Never guess a refund amount; ask if unsure").
3. **Output format** — exactly what to produce ("Reply in under 100 words. Output only the reply.").

Vague role, vague behavior. Compare the two versions below.

In [ ]:
vague  = ask("You are helpful.", "A customer was charged twice. What do I do?")
precise = ask(
    "You are a billing support specialist. Rules: acknowledge the issue, ask for the order ID if missing, "
    "never promise a specific refund amount. Output only the reply, under 60 words.",
    "A customer was charged twice. What do I do?")

print("VAGUE:\n", vague, "\n")
print("PRECISE:\n", precise)

The precise version stays in role, follows the rules, and hits the format. You steer behavior far more with a clear system prompt than with code around the call.

## Be specific, and state the output format

Models fill ambiguity with their own assumptions. Two habits fix most bad outputs:

- **Say exactly what you want**, including length, tone, and audience.
- **End with the output contract** — "Output ONLY the list", "Reply with valid JSON and nothing else". This kills the "Sure! Here's..." preamble that breaks parsing (you saw this in 0.2).

## The "rules list" pattern

For anything with constraints, a numbered/bulleted **Rules** block beats a paragraph. Models follow explicit lists more reliably than prose, and you can add or remove a rule without rewriting the prompt. This is the pattern used throughout the course's system prompts:

```
You are a <role>.
<one-line goal>

Rules:
- <constraint 1>
- <constraint 2>
- <what to do when unsure>

Output ONLY <the thing>.
```

## Few-shot: show, don't just tell

When a rule is hard to describe but easy to demonstrate, give one or two **examples** in the prompt. Few-shot examples are often more effective than another paragraph of instructions.

In [ ]:
system = (
    "Extract the sentiment as one word. Examples:\n"
    "Input: 'This is amazing!' -> positive\n"
    "Input: 'It broke on day one.' -> negative\n"
    "Input: 'It arrived on Tuesday.' -> neutral")
print(ask(system, "Honestly the best purchase I've made all year."))

## Reasoning prompts

For multi-step problems, asking the model to **work step by step** before answering improves accuracy — it's the same idea as the ReAct *reasoning* step from 0.0 and 0.3. Modern models often do this on their own, but a nudge still helps on harder tasks:

> "Think through this step by step, then give the final answer on the last line prefixed with `ANSWER:`."

For agents, the reasoning happens across the loop — the model reasons, calls a tool, sees the result, reasons again — so you rarely need to prompt for it explicitly. Where it matters is the **system prompt telling the agent *how* to approach tasks** ("Always search before answering; never guess").

## Prompt engineering for agents specifically

Two levers dominate agent behavior, and both are prompt engineering:

| Lever | What it controls | Covered in |
|-------|------------------|-----------|
| **System prompt** | The agent's overall strategy and boundaries | 1.1 |
| **Tool descriptions** | *Which* tool the agent picks and *when* | 1.2 |

A tool's docstring is a prompt the model reads on every turn — treat it like one. And remember **context engineering** (0.0, 0.6): the best prompt in the world can't help if the relevant facts aren't in front of the model. Getting the right information into the context is half of prompt engineering for agents.

## Context engineering

Prompt engineering is what you *say*; **context engineering** is what you put in front of the model *at all*, on every turn of the loop. It's repeatedly named the most under-taught, highest-impact agent skill — and it's the thread running under memory (0.6), RAG (1.7), and tool design.

The window is a fixed token budget (0.0), so spend it deliberately:

- **Include only what this step needs.** More context is not better — irrelevant text distracts the model and costs tokens. Retrieve the relevant few chunks (RAG) instead of pasting everything.
- **Compact as you go.** Summarize old turns (0.6); collapse a long tool result to the part that matters before feeding it back into the loop.
- **Truncate tool output.** A tool that returns 10,000 rows should hand the agent a head or a summary, not the whole dump.
- **Place the important thing where the model looks** — standing rules in the system prompt, the current task in the latest user turn.

A checklist for *what earns a place in the window*: is it needed for **this** decision? Is it in the **smallest** form that still works? Would removing it change the answer? If it fails the first or third test, cut it.

## Common pitfalls

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| No output contract | "Sure! Here's your JSON: ..." | End with "Output ONLY ..." + use JSON mode (0.2) |
| Conflicting rules | Model picks one and ignores the other | Remove the conflict; order rules by priority |
| Overlong prompt | Model loses the important instruction | Cut to the essential rules; move examples to few-shot |
| Negatives only | "Don't be rude" without saying what TO do | State the desired behavior positively |
| Vague tool docstring | Agent skips or misuses the tool | Say what it does, its args (with an example), and when to use it |

```{note}
Prompt engineering is empirical. Change one thing, run it, and — for anything you care about — put it in an **eval** (1.6) so "it seems better" becomes a number.
```

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Add an output contract.** Take the `vague` prompt and add role + rules + "Output only the reply", then compare.
2. **Few-shot a format.** Write a prompt that turns a sentence into a `{"subject":..., "action":...}` object using two examples, then test it on a new sentence.
3. **Break it.** Give the model two contradictory rules and watch which one it follows — a reminder to keep rules consistent.
::::